<a href="https://colab.research.google.com/github/dakshini01/Statistical-Learning-e20181/blob/main/data_analyse/data_analyse_d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
==============================================================================
ML Data Inspector — Google Colab Edition
==============================================================================
A machine-learning focused extension of DataInspector that supports:
  • CSV, Excel (.xlsx/.xls), JSON, Parquet, Feather, HDF5, Google Sheets URL
  • Full EDA + feature engineering utilities
  • Supervised ML model benchmarking (classification & regression)
  • Unsupervised clustering (KMeans, DBSCAN, Agglomerative)
  • Dimensionality reduction (PCA, t-SNE, UMAP*)
  • Feature selection (correlation filter, RFE, SHAP*)
  • Class imbalance handling (SMOTE, ADASYN, undersampling)
  • Learning curves, confusion matrices, ROC/PR curves (Plotly)
  • Auto-export of trained models (.pkl)

* Requires optional packages: umap-learn, shap
==============================================================================
"""

from __future__ import annotations

# ── Standard library ──────────────────────────────────────────────────────────
import io
import json
import os
import pickle
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

warnings.filterwarnings("ignore")

# ── Core scientific stack ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Plotly ────────────────────────────────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (
    AdaBoostClassifier, AdaBoostRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor,
    RandomForestClassifier, RandomForestRegressor,
)
from sklearn.feature_selection import RFE, SelectKBest, f_classif, f_regression
from sklearn.inspection import permutation_importance
from sklearn.linear_model import (
    ElasticNet, Lasso, LinearRegression,
    LogisticRegression, Ridge,
)
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    explained_variance_score, f1_score, mean_absolute_error,
    mean_squared_error, precision_recall_curve, r2_score,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV, KFold, StratifiedKFold,
    cross_val_score, learning_curve, train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    LabelEncoder, MinMaxScaler, RobustScaler,
    StandardScaler, label_binarize,
)
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# ── Optional imports ──────────────────────────────────────────────────────────
try:
    from xgboost import XGBClassifier, XGBRegressor
    _HAS_XGB = True
except ImportError:
    _HAS_XGB = False

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    _HAS_LGB = True
except ImportError:
    _HAS_LGB = False

try:
    from imblearn.over_sampling import SMOTE, ADASYN
    from imblearn.under_sampling import RandomUnderSampler
    _HAS_IMBALANCED = True
except ImportError:
    _HAS_IMBALANCED = False

try:
    import umap
    _HAS_UMAP = True
except ImportError:
    _HAS_UMAP = False

try:
    import shap
    _HAS_SHAP = True
except ImportError:
    _HAS_SHAP = False

# ── Google Colab utilities ────────────────────────────────────────────────────
try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

# ── scipy ─────────────────────────────────────────────────────────────────────
import scipy
from scipy.stats import chi2_contingency, f_oneway, pointbiserialr


# ==============================================================================
#  Helper: install missing optional packages inside Colab
# ==============================================================================

def install_optional(package: str):
    """pip-install a package at runtime (Colab-friendly)."""
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    print(f"✅ Installed '{package}'. Restart the kernel if imports still fail.")


# ==============================================================================
#  MLDataInspector
# ==============================================================================

class MLDataInspector:
    """
    Comprehensive ML-ready data analysis tool for Google Colab.

    Workflow
    --------
    1.  load_data()            — upload & parse any supported file format
    2.  get_summary()          — shape, dtypes, missing counts
    3.  show_missing_data()    — rows with NaNs
    4.  handle_missing_values()— impute
    5.  remove_duplicates()
    6.  handle_outliers()
    7.  encode_features()      — label / one-hot / ordinal
    8.  scale_features()       — standard / minmax / robust
    9.  select_features()      — filter / RFE / importance
    10. balance_classes()      — SMOTE / ADASYN / undersample
    11. benchmark_classifiers()/ benchmark_regressors()
    12. tune_model()           — GridSearchCV wrapper
    13. evaluate_model()       — metrics + interactive plots
    14. cluster_data()         — KMeans / DBSCAN / Agglomerative
    15. reduce_dimensions()    — PCA / t-SNE / UMAP
    16. plot_learning_curve()
    17. export_model()         — save .pkl
    18. export_cleaned_data()  — download CSV
    """

    # ── Supported file readers ───────────────────────────────────────────────
    _READERS: Dict[str, Any] = {
        ".csv":     "csv",
        ".tsv":     "tsv",
        ".txt":     "tsv",
        ".xlsx":    "excel",
        ".xls":     "excel",
        ".xlsm":    "excel",
        ".json":    "json",
        ".parquet": "parquet",
        ".feather": "feather",
        ".ftr":     "feather",
        ".h5":      "hdf",
        ".hdf":     "hdf",
        ".hdf5":    "hdf",
        ".pkl":     "pickle",
        ".pickle":  "pickle",
    }

    def __init__(self):
        self.df: Optional[pd.DataFrame] = None
        self.X_train: Optional[np.ndarray] = None
        self.X_test:  Optional[np.ndarray] = None
        self.y_train: Optional[np.ndarray] = None
        self.y_test:  Optional[np.ndarray] = None
        self._label_encoders: Dict[str, LabelEncoder] = {}
        self._scaler = None
        self._last_model = None
        self._task: Optional[str] = None   # 'classification' or 'regression'

    # ==========================================================================
    #  1. DATA LOADING — multi-format
    # ==========================================================================

    def load_data(
        self,
        filepath: Optional[str] = None,
        sheet_name: Union[str, int] = 0,
        hdf_key: str = "data",
        json_orient: str = "records",
        google_sheet_url: Optional[str] = None,
    ) -> pd.DataFrame:
        """
        Load data from a file or Google Sheets URL.

        Supported formats
        -----------------
        CSV, TSV, TXT, Excel (xlsx/xls/xlsm), JSON, Parquet,
        Feather, HDF5, Pickle, Google Sheets (public share URL).

        Parameters
        ----------
        filepath          : local path OR upload via Colab dialog if None
        sheet_name        : for Excel — sheet index or name
        hdf_key           : for HDF5 — dataset key
        json_orient       : pandas JSON orient string
        google_sheet_url  : public Google Sheets URL (exports as CSV)

        Returns
        -------
        pd.DataFrame
        """
        # ── Google Sheets URL shortcut ────────────────────────────────────────
        if google_sheet_url:
            return self._load_google_sheet(google_sheet_url)

        # ── Colab file upload dialog ──────────────────────────────────────────
        if filepath is None:
            if not _IN_COLAB:
                raise RuntimeError("filepath is required outside Google Colab.")
            print("📁 Please upload your data file …")
            uploaded = files.upload()
            if not uploaded:
                print("⚠️  No file uploaded.")
                return pd.DataFrame()
            filename = list(uploaded.keys())[0]
            raw_bytes = uploaded[filename]
            return self._parse_bytes(filename, io.BytesIO(raw_bytes), sheet_name, hdf_key, json_orient)

        # ── Local / Drive path ────────────────────────────────────────────────
        filepath = Path(filepath)
        if not filepath.exists():
            raise FileNotFoundError(f"File not found: {filepath}")
        return self._parse_bytes(filepath.name, open(filepath, "rb"), sheet_name, hdf_key, json_orient)

    def _load_google_sheet(self, url: str) -> pd.DataFrame:
        """Convert a Google Sheets share URL to a CSV export URL and read it."""
        import re
        # Transform /edit?... → /export?format=csv
        match = re.search(r"/d/([a-zA-Z0-9_-]+)", url)
        if not match:
            raise ValueError("Could not parse Google Sheets ID from URL.")
        sheet_id = match.group(1)
        gid_match = re.search(r"gid=(\d+)", url)
        gid = gid_match.group(1) if gid_match else "0"
        export_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
        print(f"🔗 Fetching Google Sheet → {export_url}")
        self.df = pd.read_csv(export_url, na_values=["?", "n/a", "N/A", "NULL", "null", " "])
        self._post_load()
        print(f"✅ Google Sheet loaded — {self.df.shape[0]} rows × {self.df.shape[1]} cols")
        return self.df

    def _parse_bytes(
        self,
        filename: str,
        stream,
        sheet_name,
        hdf_key: str,
        json_orient: str,
    ) -> pd.DataFrame:
        ext = Path(filename).suffix.lower()
        fmt = self._READERS.get(ext)

        null_vals = ["?", "n/a", "N/A", "NULL", "null", " ", "NA", "nan", "NaN"]

        if fmt == "csv":
            self.df = pd.read_csv(stream, na_values=null_vals)
        elif fmt == "tsv":
            self.df = pd.read_csv(stream, sep="\t", na_values=null_vals)
        elif fmt == "excel":
            self.df = pd.read_excel(stream, sheet_name=sheet_name, na_values=null_vals)
        elif fmt == "json":
            content = stream.read()
            try:
                self.df = pd.read_json(io.BytesIO(content), orient=json_orient)
            except Exception:
                # fallback: try records list
                data = json.loads(content)
                if isinstance(data, list):
                    self.df = pd.DataFrame(data)
                elif isinstance(data, dict):
                    self.df = pd.DataFrame([data])
                else:
                    raise ValueError("Unsupported JSON structure.")
        elif fmt == "parquet":
            self.df = pd.read_parquet(stream)
        elif fmt == "feather":
            self.df = pd.read_feather(stream)
        elif fmt == "hdf":
            self.df = pd.read_hdf(stream, key=hdf_key)
        elif fmt == "pickle":
            obj = pickle.load(stream)
            self.df = pd.DataFrame(obj) if not isinstance(obj, pd.DataFrame) else obj
        else:
            raise ValueError(
                f"Unsupported extension '{ext}'.\n"
                f"Supported: {', '.join(self._READERS.keys())}"
            )

        self._post_load()
        print(f"✅ '{filename}' loaded — {self.df.shape[0]} rows × {self.df.shape[1]} cols")
        return self.df

    def _post_load(self):
        """Auto-coerce numeric-looking columns and add helper count column."""
        for col in self.df.columns:
            coerced = pd.to_numeric(self.df[col], errors="coerce")
            if not coerced.isna().all():
                self.df[col] = coerced
        if "count" not in self.df.columns:
            self.df["count"] = 1

    # ==========================================================================
    #  2. BASIC INSPECTION
    # ==========================================================================

    def get_summary(self):
        """Print shape, dtypes, missing values, and display first 20 rows."""
        if self.df is None:
            return print("❌ No data loaded.")
        num_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()
        miss = self.df.isnull().sum()
        miss = miss[miss > 0]
        print("─" * 60)
        print(f"  Rows : {self.df.shape[0]:,}   Columns : {self.df.shape[1]}")
        print(f"  Numeric  ({len(num_cols)}) : {num_cols}")
        print(f"  Categoric({len(cat_cols)}) : {cat_cols}")
        if not miss.empty:
            print(f"\n  Missing value counts:\n{miss.to_string()}")
        else:
            print("\n  ✨ No missing values.")
        print("─" * 60)
        try:
            from IPython.display import display
            display(self.df.head(20))
        except Exception:
            print(self.df.head(20))

    def show_missing_data(self):
        """Display only rows that contain at least one NaN."""
        if self.df is None:
            return
        mask = self.df.isnull().any(axis=1)
        rows = self.df[mask]
        if rows.empty:
            print("✨ No missing data found!")
        else:
            print(f"🔍 {len(rows)} rows with missing values:")
            try:
                from IPython.display import display; display(rows)
            except Exception:
                print(rows)

    def column_details(self):
        """Print range / unique-count for every column."""
        if self.df is None:
            return
        for col in self.df.columns:
            if pd.api.types.is_numeric_dtype(self.df[col]):
                print(f"🔹 {col} (Numeric)   : [{self.df[col].min():.4g} … {self.df[col].max():.4g}]")
            else:
                print(f"🔸 {col} (Categorical): {self.df[col].nunique()} unique values")

    # ==========================================================================
    #  3. DATA CLEANING
    # ==========================================================================

    def handle_missing_values(
        self,
        columns: Optional[List[str]] = None,
        strategy: str = "median",
        fill_value=None,
    ):
        """
        Impute missing values.

        strategy : 'mean' | 'median' | 'mode' | 'constant' | 'ffill' | 'bfill'
        """
        if self.df is None:
            return
        targets = columns or self.df.columns[self.df.isnull().any()].tolist()
        for col in targets:
            if strategy == "mean" and pd.api.types.is_numeric_dtype(self.df[col]):
                self.df[col].fillna(self.df[col].mean(), inplace=True)
            elif strategy == "median" and pd.api.types.is_numeric_dtype(self.df[col]):
                self.df[col].fillna(self.df[col].median(), inplace=True)
            elif strategy == "mode":
                self.df[col].fillna(self.df[col].mode()[0], inplace=True)
            elif strategy == "constant":
                self.df[col].fillna(fill_value, inplace=True)
            elif strategy == "ffill":
                self.df[col].ffill(inplace=True)
            elif strategy == "bfill":
                self.df[col].bfill(inplace=True)
        print(f"🛠️  Imputation '{strategy}' applied to: {targets}")

    def remove_duplicates(self):
        """Drop exact duplicate rows."""
        if self.df is None:
            return
        before = len(self.df)
        self.df.drop_duplicates(inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        print(f"✨ Removed {before - len(self.df)} duplicates. Rows now: {len(self.df):,}")

    def handle_outliers(
        self,
        columns: Optional[List[str]] = None,
        method: str = "iqr",
        action: str = "flag",
        z_threshold: float = 3.0,
    ):
        """
        Detect and optionally remove outliers.

        method : 'iqr' | 'zscore'
        action : 'flag' (print only) | 'remove'
        """
        if self.df is None:
            return
        targets = columns or self.df.select_dtypes(include=[np.number]).columns.tolist()
        all_outliers: set = set()

        for col in targets:
            if method == "iqr":
                q1, q3 = self.df[col].quantile(0.25), self.df[col].quantile(0.75)
                iqr = q3 - q1
                mask = (self.df[col] < q1 - 1.5 * iqr) | (self.df[col] > q3 + 1.5 * iqr)
            else:  # zscore
                zscores = (self.df[col] - self.df[col].mean()) / self.df[col].std()
                mask = zscores.abs() > z_threshold

            idxs = self.df.index[mask].tolist()
            all_outliers.update(idxs)
            print(f"🚨 {col}: {len(idxs)} outliers detected ({method.upper()})")

        if all_outliers and action == "remove":
            self.df.drop(index=list(all_outliers), inplace=True)
            self.df.reset_index(drop=True, inplace=True)
            print(f"🗑️  Removed {len(all_outliers)} outlier rows.")

    def delete_columns(self, columns: Optional[List[str]] = None):
        """Drop columns by name list (or interactive input if None)."""
        if self.df is None:
            return
        if columns is None:
            print(f"Columns: {list(self.df.columns)}")
            raw = input("Enter column names to delete (comma-separated): ")
            columns = [c.strip() for c in raw.split(",")]
        existing = [c for c in columns if c in self.df.columns]
        self.df.drop(columns=existing, inplace=True)
        print(f"🗑️  Deleted: {existing}")

    # ==========================================================================
    #  4. FEATURE ENGINEERING
    # ==========================================================================

    def encode_features(
        self,
        columns: Optional[List[str]] = None,
        method: str = "label",
    ) -> pd.DataFrame:
        """
        Encode categorical columns.

        method : 'label'  — LabelEncoder (each column independently)
                 'onehot' — pd.get_dummies (adds new binary columns)
                 'ordinal'— integer codes 0…n-1
        """
        if self.df is None:
            return pd.DataFrame()
        targets = columns or self.df.select_dtypes(exclude=[np.number]).columns.tolist()
        # remove 'count' helper if present
        targets = [c for c in targets if c != "count"]

        if method == "label":
            for col in targets:
                le = LabelEncoder()
                self.df[col] = le.fit_transform(self.df[col].astype(str))
                self._label_encoders[col] = le
            print(f"🏷️  Label-encoded: {targets}")

        elif method == "onehot":
            self.df = pd.get_dummies(self.df, columns=targets, drop_first=False)
            print(f"🔢 One-hot encoded: {targets}  →  {self.df.shape[1]} total columns")

        elif method == "ordinal":
            for col in targets:
                self.df[col] = self.df[col].astype("category").cat.codes
            print(f"🔢 Ordinal encoded: {targets}")

        return self.df

    def scale_features(
        self,
        columns: Optional[List[str]] = None,
        method: str = "standard",
    ) -> pd.DataFrame:
        """
        Scale numeric columns.

        method : 'standard' | 'minmax' | 'robust'
        """
        if self.df is None:
            return pd.DataFrame()
        targets = columns or [
            c for c in self.df.select_dtypes(include=[np.number]).columns
            if c != "count"
        ]
        scaler_map = {
            "standard": StandardScaler(),
            "minmax":   MinMaxScaler(),
            "robust":   RobustScaler(),
        }
        scaler = scaler_map.get(method, StandardScaler())
        self.df[targets] = scaler.fit_transform(self.df[targets].fillna(0))
        self._scaler = scaler
        print(f"⚖️  Scaled ({method}): {targets}")
        return self.df

    def add_polynomial_features(
        self,
        columns: List[str],
        degree: int = 2,
    ) -> pd.DataFrame:
        """
        Add polynomial interaction terms for selected numeric columns.
        New columns named <col>^<degree> are appended.
        """
        if self.df is None:
            return pd.DataFrame()
        for col in columns:
            if pd.api.types.is_numeric_dtype(self.df[col]):
                for d in range(2, degree + 1):
                    self.df[f"{col}^{d}"] = self.df[col] ** d
        print(f"🧮 Polynomial features (degree={degree}) added for: {columns}")
        return self.df

    def add_log_features(self, columns: List[str]) -> pd.DataFrame:
        """
        Add log1p-transformed versions of numeric columns.
        Useful for skewed distributions.
        """
        if self.df is None:
            return pd.DataFrame()
        for col in columns:
            if pd.api.types.is_numeric_dtype(self.df[col]):
                self.df[f"log_{col}"] = np.log1p(np.clip(self.df[col], 0, None))
        print(f"📐 Log features added for: {columns}")
        return self.df

    # ==========================================================================
    #  5. FEATURE SELECTION
    # ==========================================================================

    def select_features(
        self,
        target: str,
        method: str = "correlation",
        k: int = 10,
        task: str = "classification",
        estimator=None,
    ) -> List[str]:
        """
        Select the top-k features using various methods.

        method    : 'correlation' | 'univariate' | 'rfe' | 'importance'
        task      : 'classification' | 'regression'
        estimator : optional sklearn estimator for 'rfe' / 'importance'
        """
        if self.df is None or target not in self.df.columns:
            print("❌ DataFrame or target column missing.")
            return []

        feature_cols = [
            c for c in self.df.select_dtypes(include=[np.number]).columns
            if c not in (target, "count")
        ]
        X = self.df[feature_cols].fillna(0).values
        y = self.df[target].values

        selected: List[str] = []

        if method == "correlation":
            corrs = self.df[feature_cols].corrwith(self.df[target]).abs()
            selected = corrs.nlargest(k).index.tolist()

        elif method == "univariate":
            score_fn = f_classif if task == "classification" else f_regression
            sel = SelectKBest(score_fn, k=min(k, len(feature_cols)))
            sel.fit(X, y)
            selected = [feature_cols[i] for i in sel.get_support(indices=True)]

        elif method == "rfe":
            est = estimator or (
                RandomForestClassifier(n_estimators=50, random_state=42)
                if task == "classification"
                else RandomForestRegressor(n_estimators=50, random_state=42)
            )
            rfe = RFE(est, n_features_to_select=min(k, len(feature_cols)))
            rfe.fit(X, y)
            selected = [feature_cols[i] for i in range(len(feature_cols)) if rfe.support_[i]]

        elif method == "importance":
            est = estimator or (
                RandomForestClassifier(n_estimators=100, random_state=42)
                if task == "classification"
                else RandomForestRegressor(n_estimators=100, random_state=42)
            )
            est.fit(X, y)
            importances = pd.Series(est.feature_importances_, index=feature_cols)
            selected = importances.nlargest(k).index.tolist()
            # Plot
            fig = px.bar(
                importances.nlargest(k).reset_index(),
                x="index", y=0, title=f"Top-{k} Feature Importances",
                labels={"index": "Feature", 0: "Importance"},
                color=0, color_continuous_scale="Viridis",
            )
            fig.show()

        print(f"🎯 Top-{k} features ({method}): {selected}")
        return selected

    # ==========================================================================
    #  6. CLASS IMBALANCE
    # ==========================================================================

    def balance_classes(
        self,
        X: np.ndarray,
        y: np.ndarray,
        method: str = "smote",
        random_state: int = 42,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Resample an imbalanced dataset.

        method : 'smote' | 'adasyn' | 'undersample'

        Requires imbalanced-learn:
            pip install imbalanced-learn
        """
        if not _HAS_IMBALANCED:
            print("⚠️  imbalanced-learn not installed. Run: pip install imbalanced-learn")
            return X, y

        unique, counts = np.unique(y, return_counts=True)
        print(f"📊 Before resampling: {dict(zip(unique, counts))}")

        if method == "smote":
            sampler = SMOTE(random_state=random_state)
        elif method == "adasyn":
            sampler = ADASYN(random_state=random_state)
        else:
            sampler = RandomUnderSampler(random_state=random_state)

        X_res, y_res = sampler.fit_resample(X, y)
        unique2, counts2 = np.unique(y_res, return_counts=True)
        print(f"📊 After  resampling: {dict(zip(unique2, counts2))}")
        return X_res, y_res

    # ==========================================================================
    #  7. TRAIN / TEST SPLIT
    # ==========================================================================

    def prepare_train_test(
        self,
        target: str,
        feature_cols: Optional[List[str]] = None,
        test_size: float = 0.2,
        random_state: int = 42,
        stratify: bool = True,
    ):
        """
        Split the DataFrame into train/test arrays stored on self.

        Returns (X_train, X_test, y_train, y_test)
        """
        if self.df is None or target not in self.df.columns:
            raise ValueError("DataFrame or target column missing.")

        num_cols = [
            c for c in self.df.select_dtypes(include=[np.number]).columns
            if c not in (target, "count")
        ]
        features = feature_cols or num_cols
        X = self.df[features].fillna(0).values
        y = self.df[target].values

        strat = y if stratify else None
        try:
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                X, y, test_size=test_size, random_state=random_state, stratify=strat
            )
        except ValueError:
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                X, y, test_size=test_size, random_state=random_state
            )

        print(
            f"✂️  Train: {self.X_train.shape}  Test: {self.X_test.shape}  "
            f"Target: '{target}'"
        )
        return self.X_train, self.X_test, self.y_train, self.y_test

    # ==========================================================================
    #  8. MODEL BENCHMARKING
    # ==========================================================================

    def benchmark_classifiers(
        self,
        cv: int = 5,
        scoring: str = "accuracy",
        show_plot: bool = True,
    ) -> pd.DataFrame:
        """
        Cross-validate a suite of classifiers on (X_train, y_train).
        Plots a horizontal bar chart of mean CV scores.
        """
        if self.X_train is None:
            print("❌ Run prepare_train_test() first.")
            return pd.DataFrame()

        self._task = "classification"
        models = {
            "Logistic Regression":   LogisticRegression(max_iter=1000),
            "Decision Tree":         DecisionTreeClassifier(),
            "Random Forest":         RandomForestClassifier(n_estimators=100),
            "Gradient Boosting":     GradientBoostingClassifier(),
            "AdaBoost":              AdaBoostClassifier(),
            "K-Nearest Neighbors":   KNeighborsClassifier(),
            "SVM (RBF)":             SVC(probability=True),
            "Naive Bayes":           GaussianNB(),
        }
        if _HAS_XGB:
            models["XGBoost"] = XGBClassifier(eval_metric="logloss", verbosity=0)
        if _HAS_LGB:
            models["LightGBM"] = LGBMClassifier(verbose=-1)

        results = []
        for name, model in models.items():
            cv_scores = cross_val_score(
                model, self.X_train, self.y_train,
                cv=StratifiedKFold(n_splits=cv, shuffle=True, random_state=42),
                scoring=scoring,
            )
            results.append({
                "Model": name,
                "Mean": cv_scores.mean(),
                "Std":  cv_scores.std(),
            })
            print(f"  {name:30s}  {scoring}: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

        results_df = results_df_raw = pd.DataFrame(results).sort_values("Mean", ascending=False)

        if show_plot:
            fig = px.bar(
                results_df,
                x="Mean", y="Model", orientation="h",
                error_x="Std",
                title=f"Classifier Benchmark — {scoring} (CV={cv})",
                color="Mean", color_continuous_scale="RdYlGn",
                labels={"Mean": scoring},
            )
            fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=500)
            fig.show()

        return results_df

    def benchmark_regressors(
        self,
        cv: int = 5,
        scoring: str = "r2",
        show_plot: bool = True,
    ) -> pd.DataFrame:
        """
        Cross-validate a suite of regressors on (X_train, y_train).
        """
        if self.X_train is None:
            print("❌ Run prepare_train_test() first.")
            return pd.DataFrame()

        self._task = "regression"
        models = {
            "Linear Regression":     LinearRegression(),
            "Ridge":                 Ridge(),
            "Lasso":                 Lasso(),
            "ElasticNet":            ElasticNet(),
            "Decision Tree":         DecisionTreeRegressor(),
            "Random Forest":         RandomForestRegressor(n_estimators=100),
            "Gradient Boosting":     GradientBoostingRegressor(),
            "AdaBoost":              AdaBoostRegressor(),
            "K-Nearest Neighbors":   KNeighborsRegressor(),
            "SVR":                   SVR(),
        }
        if _HAS_XGB:
            models["XGBoost"] = XGBRegressor(verbosity=0)
        if _HAS_LGB:
            models["LightGBM"] = LGBMRegressor(verbose=-1)

        results = []
        for name, model in models.items():
            cv_scores = cross_val_score(
                model, self.X_train, self.y_train,
                cv=KFold(n_splits=cv, shuffle=True, random_state=42),
                scoring=scoring,
            )
            results.append({"Model": name, "Mean": cv_scores.mean(), "Std": cv_scores.std()})
            print(f"  {name:30s}  {scoring}: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

        results_df = pd.DataFrame(results).sort_values("Mean", ascending=False)

        if show_plot:
            fig = px.bar(
                results_df,
                x="Mean", y="Model", orientation="h",
                error_x="Std",
                title=f"Regressor Benchmark — {scoring} (CV={cv})",
                color="Mean", color_continuous_scale="RdYlGn",
                labels={"Mean": scoring},
            )
            fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=500)
            fig.show()

        return results_df

    # ==========================================================================
    #  9. HYPERPARAMETER TUNING
    # ==========================================================================

    def tune_model(
        self,
        model,
        param_grid: Dict,
        cv: int = 5,
        scoring: str = "accuracy",
        n_jobs: int = -1,
    ):
        """
        GridSearchCV wrapper. Returns the fitted GridSearchCV object.

        Example
        -------
        grid = inspector.tune_model(
            RandomForestClassifier(),
            {"n_estimators": [50, 100, 200], "max_depth": [None, 5, 10]},
        )
        print(grid.best_params_)
        """
        if self.X_train is None:
            print("❌ Run prepare_train_test() first.")
            return None

        grid = GridSearchCV(model, param_grid, cv=cv, scoring=scoring,
                            n_jobs=n_jobs, verbose=1)
        grid.fit(self.X_train, self.y_train)
        print(f"\n🏆 Best params : {grid.best_params_}")
        print(f"   Best {scoring}: {grid.best_score_:.4f}")
        self._last_model = grid.best_estimator_
        return grid

    # ==========================================================================
    #  newly added codes
    # ==========================================================================

    def correlate_num_to_cat(self):
        """Find association between numeric and categorical columns."""
        if self.df is None:
            print("❌ No data loaded.")
            return pd.DataFrame()

        num_cols = [c for c in self.df.select_dtypes(include=[np.number]).columns if c != "count"]
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()

        if not num_cols or not cat_cols:
            print("⚠️ Requires both numerical and categorical columns.")
            return pd.DataFrame()

        results = []
        for cat in cat_cols:
            for num in num_cols:
                valid = self.df[[cat, num]].dropna()
                if valid.empty or valid[cat].nunique() < 2:
                    continue

                if valid[cat].nunique() == 2:
                    binary = pd.get_dummies(valid[cat], drop_first=True).iloc[:, 0]
                    corr, p_val = pointbiserialr(binary, valid[num])
                    method = "Point-Biserial"
                    strength = abs(corr)
                else:
                    groups = [valid.loc[valid[cat] == v, num] for v in valid[cat].unique()]
                    groups = [g for g in groups if len(g) > 0]
                    _, p_val = f_oneway(*groups)
                    grand_mean = valid[num].mean()
                    ss_total = ((valid[num] - grand_mean) ** 2).sum()
                    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
                    strength = np.sqrt(ss_between / ss_total) if ss_total > 0 else 0.0
                    corr = strength
                    method = "Eta / ANOVA"

                results.append({
                    "Categorical": cat,
                    "Numerical": num,
                    "Method": method,
                    "Association": round(float(strength), 4),
                    "Signed_Correlation": round(float(corr), 4),
                    "P_Value": round(float(p_val), 6),
                })

        out = pd.DataFrame(results).sort_values("Association", ascending=False)
        try:
            display(out)
        except Exception:
            print(out)
        return out


    def plot_all_associations_heatmap(self):
        """Unified association heatmap for numeric-numeric, categorical-categorical, and numeric-categorical pairs."""
        if self.df is None:
            print("❌ No data loaded.")
            return pd.DataFrame()

        cols = [c for c in self.df.columns if c != "count"]
        matrix = pd.DataFrame(np.eye(len(cols)), index=cols, columns=cols)

        for i, c1 in enumerate(cols):
            for j, c2 in enumerate(cols[i + 1:], start=i + 1):
                valid = self.df[[c1, c2]].dropna()
                if valid.empty:
                    val = 0.0
                else:
                    n1 = pd.api.types.is_numeric_dtype(valid[c1])
                    n2 = pd.api.types.is_numeric_dtype(valid[c2])

                    if n1 and n2:
                        val = abs(valid[c1].corr(valid[c2]))
                        val = 0.0 if pd.isna(val) else val
                    elif (not n1) and (not n2):
                        tab = pd.crosstab(valid[c1], valid[c2])
                        if min(tab.shape) <= 1:
                            val = 0.0
                        else:
                            chi2 = chi2_contingency(tab)[0]
                            n = tab.to_numpy().sum()
                            val = np.sqrt(chi2 / (n * (min(tab.shape) - 1))) if n > 0 else 0.0
                    else:
                        cat_col, num_col = (c1, c2) if not n1 else (c2, c1)
                        groups = [valid.loc[valid[cat_col] == v, num_col] for v in valid[cat_col].unique()]
                        grand_mean = valid[num_col].mean()
                        ss_total = ((valid[num_col] - grand_mean) ** 2).sum()
                        ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups if len(g) > 0)
                        val = np.sqrt(ss_between / ss_total) if ss_total > 0 else 0.0

                matrix.loc[c1, c2] = round(float(val), 3)
                matrix.loc[c2, c1] = round(float(val), 3)

        fig = px.imshow(matrix, text_auto=".2f", aspect="auto", color_continuous_scale="Viridis",
                        title="Unified Association Heatmap")
        fig.show()
        return matrix


    def compute_and_plot_conditional_normal(self, x_columns: Sequence[str], y_columns: Sequence[str], show_plots: bool = True) -> Dict[str, Any]:
        """Estimate E[Y|X] assuming joint normal data; equivalent to linear regression diagnostics."""
        if self.df is None:
            raise ValueError("No data loaded.")
        x_columns, y_columns = list(x_columns), list(y_columns)
        df = self.df[x_columns + y_columns].apply(pd.to_numeric, errors="coerce").dropna()
        if df.empty:
            raise ValueError("No valid rows after dropping NaNs.")

        X, Y = df[x_columns].to_numpy(float), df[y_columns].to_numpy(float)
        n, p = X.shape
        if n <= p + 1:
            raise ValueError("Need more rows than X variables + 1.")

        mu_X, mu_Y = X.mean(axis=0), Y.mean(axis=0)
        Xc, Yc = X - mu_X, Y - mu_Y
        dof = n - 1
        Sxx = (Xc.T @ Xc) / dof
        Sxy = (Xc.T @ Yc) / dof
        Syx = Sxy.T
        Syy = (Yc.T @ Yc) / dof
        Sxx_inv = np.linalg.pinv(Sxx)
        B = Sxx_inv @ Sxy
        Y_hat = mu_Y + Xc @ B
        residuals = Y - Y_hat
        S_y_given_x = Syy - Syx @ Sxx_inv @ Sxy
        S_y_given_x = 0.5 * (S_y_given_x + S_y_given_x.T)

        reg = LinearRegression().fit(X, Y)
        rmse = np.sqrt(mean_squared_error(Y, Y_hat, multioutput="raw_values"))
        mae = mean_absolute_error(Y, Y_hat, multioutput="raw_values")
        r2 = r2_score(Y, Y_hat, multioutput="raw_values")
        metrics = pd.DataFrame({"RMSE": rmse, "MAE": mae, "R2": r2}, index=y_columns)

        if show_plots:
            px.imshow(S_y_given_x, x=y_columns, y=y_columns, text_auto=".3g",
                      title="Conditional Covariance of Y given X").show()
            for i, col in enumerate(y_columns):
                fig = px.scatter(x=Y[:, i], y=Y_hat[:, i], labels={"x": f"Actual {col}", "y": f"Predicted {col}"},
                                title=f"Observed vs Predicted: {col}")
                mn, mx = min(Y[:, i].min(), Y_hat[:, i].min()), max(Y[:, i].max(), Y_hat[:, i].max())
                fig.add_trace(go.Scatter(x=[mn, mx], y=[mn, mx], mode="lines", name="Ideal"))
                fig.show()
            px.bar(metrics.reset_index().melt(id_vars="index"), x="index", y="value", color="variable",
                  barmode="group", title="Prediction Metrics").show()

        return {"mu_X": mu_X, "mu_Y": mu_Y, "Sigma_XX": Sxx, "Sigma_YY": Syy, "Sigma_Y_given_X": S_y_given_x,
                "conditional_coefficient_matrix": B, "conditional_intercept": mu_Y - mu_X @ B,
                "Y_hat": Y_hat, "residuals": residuals, "metrics_dataframe": metrics,
                "sklearn_model_object": reg, "x_features": x_columns, "y_features": y_columns}


    def compute_empirical_pca_advanced(self, columns: Optional[Sequence[str]] = None, show_plot: bool = True) -> Dict[str, Any]:
        """Advanced PCA with loadings, explained variance, Hotelling T² and Q/SPE profiles."""
        if self.df is None:
            raise ValueError("No data loaded.")
        cols = list(columns) if columns is not None else [c for c in self.df.select_dtypes(include=[np.number]).columns if c != "count"]
        X = self.df[cols].dropna().to_numpy(float)
        n, m = X.shape
        if n <= m:
            raise ValueError("Sample size must be larger than number of features.")

        mu = X.mean(axis=0)
        Xc = X - mu
        S = np.cov(X, rowvar=False, ddof=1)
        eigvals, eigvecs = np.linalg.eigh(S)
        idx = np.argsort(eigvals)[::-1]
        lam = np.clip(eigvals[idx], 1e-15, None)
        P = eigvecs[:, idx]
        Z = Xc @ P
        evr = lam / lam.sum()
        cum_evr = np.cumsum(evr)
        k_range = np.arange(1, m)
        T2 = np.zeros((n, len(k_range)))
        Q = np.zeros((n, len(k_range)))
        for i, k in enumerate(k_range):
            T2[:, i] = np.sum((Z[:, :k] ** 2) / lam[:k], axis=1)
            Q[:, i] = np.sum(Z[:, k:] ** 2, axis=1)

        if show_plot:
            pcs = [f"PC{i+1}" for i in range(m)]
            fig = make_subplots(rows=2, cols=2, subplot_titles=("Loading Matrix |P|", "Explained Variance",
                                                                "Mean Hotelling T²", "Mean Q/SPE"))
            fig.add_trace(go.Heatmap(z=np.abs(P), x=pcs, y=cols, colorscale="Viridis"), row=1, col=1)
            fig.add_trace(go.Bar(x=pcs, y=evr * 100, name="Individual %"), row=1, col=2)
            fig.add_trace(go.Scatter(x=pcs, y=cum_evr * 100, mode="lines+markers", name="Cumulative %"), row=1, col=2)
            fig.add_trace(go.Scatter(x=[f"k={k}" for k in k_range], y=T2.mean(axis=0), mode="lines+markers"), row=2, col=1)
            fig.add_trace(go.Scatter(x=[f"k={k}" for k in k_range], y=Q.mean(axis=0), mode="lines+markers"), row=2, col=2)
            fig.update_layout(title="Advanced PCA Dashboard", height=750, template="plotly_white")
            fig.show()

        return {"mean_vector": mu, "covariance_matrix_S": S, "eigenvalues_lambda": lam,
                "eigenvectors_P": P, "explained_variance_ratio": evr,
                "cumulative_variance_ratio": cum_evr, "transformed_scores_Z": Z,
                "k_values": k_range, "T2_matrix_vs_k": T2, "Q_matrix_vs_k": Q, "features": cols}


    def compute_empirical_fa(self, k: int, columns: Optional[Sequence[str]] = None, show_plot: bool = True) -> Dict[str, Any]:
        """Factor Analysis to find hidden factors behind numeric variables."""
        if self.df is None:
            raise ValueError("No data loaded.")
        cols = list(columns) if columns is not None else [c for c in self.df.select_dtypes(include=[np.number]).columns if c != "count"]
        X = self.df[cols].dropna().to_numpy(float)
        n, m = X.shape
        if k >= m:
            raise ValueError("k must be smaller than number of features.")
        if n <= m:
            raise ValueError("Sample size must be larger than feature count.")

        mu = X.mean(axis=0)
        std = X.std(axis=0, ddof=1)
        std[std == 0] = 1e-15
        Z = (X - mu) / std
        fa = FactorAnalysis(n_components=k, rotation="varimax", random_state=42).fit(Z)
        loadings = fa.components_.T
        uniqueness = fa.noise_variance_
        communality = np.sum(loadings ** 2, axis=1)
        scores = fa.transform(Z)

        if show_plot:
            factors = [f"Factor {i+1}" for i in range(k)]
            fig = make_subplots(rows=1, cols=2, subplot_titles=("Factor Loadings |λ|", "Communality vs Uniqueness"))
            fig.add_trace(go.Heatmap(z=np.abs(loadings), x=factors, y=cols, colorscale="Viridis"), row=1, col=1)
            fig.add_trace(go.Bar(y=cols, x=communality, orientation="h", name="Communality"), row=1, col=2)
            fig.add_trace(go.Bar(y=cols, x=uniqueness, orientation="h", name="Uniqueness"), row=1, col=2)
            fig.update_layout(title="Factor Analysis Dashboard", barmode="stack", height=550, template="plotly_white")
            fig.show()

        return {"mean_vector_mu": mu, "std_vector": std, "factor_loadings_lambda": loadings,
                "uniqueness_psi": uniqueness, "communality_h2": communality,
                "latent_factor_scores_F": scores, "features": cols}


    def test_constant_mean(self, columns: Optional[Sequence[str]] = None, chunks: int = 10) -> Dict[str, Any]:
        """Check whether numeric column means stay stable across row chunks using Wilks' Lambda approximation."""
        if self.df is None:
            raise ValueError("No data loaded.")
        cols = list(columns) if columns is not None else [c for c in self.df.select_dtypes(include=[np.number]).columns if c != "count"]
        Xdf = self.df[cols].dropna()
        n, m = Xdf.shape
        chunk_size = n // chunks
        if chunk_size <= m:
            raise ValueError("Reduce chunks or use fewer columns.")
        Xdf = Xdf.copy()
        Xdf["_chunk"] = np.minimum(np.arange(n) // chunk_size, chunks - 1)
        global_mean = Xdf[cols].mean().to_numpy()
        W = np.zeros((m, m)); B = np.zeros((m, m))
        for _, g in Xdf.groupby("_chunk"):
            X = g[cols].to_numpy(float)
            cm = X.mean(axis=0); nj = len(X)
            W += (X - cm).T @ (X - cm)
            d = (cm - global_mean).reshape(-1, 1)
            B += nj * (d @ d.T)
        eps = 1e-6 * np.eye(m)
        log_w = np.linalg.slogdet(W + eps)[1]
        log_t = np.linalg.slogdet(W + B + eps)[1]
        log_lambda = log_w - log_t
        wilks = float(np.exp(log_lambda))
        df_stat = m * (chunks - 1)
        chi2 = max(0.0, float(-(n - 1 - (m + chunks) / 2) * log_lambda))
        p = float(1 - scipy.stats.chi2.cdf(chi2, df_stat))
        print(f"Wilks Lambda={wilks:.5f}, Chi2={chi2:.4f}, p={p:.6f}")
        print("✅ Mean stable" if p > 0.05 else "🚨 Mean drift detected")
        return {"wilks_lambda": wilks, "chi2": chi2, "p_value": p, "df": df_stat}


    def test_constant_covariance(self, columns: Optional[Sequence[str]] = None, chunks: int = 5) -> Dict[str, Any]:
        """Check whether covariance structure stays stable across row chunks using Box's M approximation."""
        if self.df is None:
            raise ValueError("No data loaded.")
        cols = list(columns) if columns is not None else [c for c in self.df.select_dtypes(include=[np.number]).columns if c != "count"]
        Xdf = self.df[cols].dropna()
        n, m = Xdf.shape
        chunk_size = n // chunks
        if chunk_size <= m:
            raise ValueError("Reduce chunks or use fewer columns.")
        Xdf = Xdf.copy()
        Xdf["_chunk"] = np.minimum(np.arange(n) // chunk_size, chunks - 1)
        eps = 1e-6 * np.eye(m)
        pooled = np.zeros((m, m)); total_df = 0; log_det_sum = 0; ns = []
        for _, g in Xdf.groupby("_chunk"):
            X = g[cols].to_numpy(float); nj = len(X); ns.append(nj)
            S = np.cov(X, rowvar=False, ddof=1) + eps
            dfj = nj - 1
            pooled += dfj * S; total_df += dfj
            log_det_sum += dfj * np.linalg.slogdet(S)[1]
        pooled /= total_df
        M = total_df * np.linalg.slogdet(pooled)[1] - log_det_sum
        C = (sum(1/(nj-1) for nj in ns) - 1/total_df) * ((2*m*m + 3*m - 1)/(6*(m+1)*(chunks-1)))
        chi2 = max(0.0, float(M * (1 - C)))
        df_stat = int(m * (m + 1) * (chunks - 1) / 2)
        p = float(1 - scipy.stats.chi2.cdf(chi2, df_stat))
        print(f"Box M={M:.4f}, Chi2={chi2:.4f}, p={p:.6f}")
        print("✅ Covariance stable" if p > 0.001 else "🚨 Covariance drift detected")
        return {"M": float(M), "chi2": chi2, "p_value": p, "df": df_stat}


    def test_row_independence(self, columns: Optional[Sequence[str]] = None, max_lag: Optional[int] = None) -> Dict[str, Any]:
        """Check row-to-row serial dependence using a multivariate Ljung-Box style statistic."""
        if self.df is None:
            raise ValueError("No data loaded.")
        cols = list(columns) if columns is not None else [c for c in self.df.select_dtypes(include=[np.number]).columns if c != "count"]
        X = self.df[cols].dropna().to_numpy(float)
        n, m = X.shape
        max_lag = int(np.ceil(np.log(n))) if max_lag is None else max_lag
        if max_lag >= n:
            raise ValueError("max_lag must be smaller than sample size.")
        Xc = X - X.mean(axis=0)
        G0 = (Xc.T @ Xc) / n + 1e-6 * np.eye(m)
        invG0 = np.linalg.pinv(G0)
        Q = 0.0
        for k in range(1, max_lag + 1):
            Gk = Xc[k:].T @ Xc[:-k] / n
            Q += np.trace(Gk.T @ invG0 @ Gk @ invG0) / (n - k)
        Q = float(max(0.0, Q * n * n))
        df_stat = m * m * max_lag
        p = float(1 - scipy.stats.chi2.cdf(Q, df_stat))
        print(f"Q={Q:.4f}, df={df_stat}, p={p:.6f}")
        print("✅ Rows look independent" if p > 0.05 else "🚨 Serial dependency detected")
        return {"Q_m": Q, "p_value": p, "df": df_stat}


    def estimate_joint_normal(self, columns: Optional[Sequence[str]] = None) -> Dict[str, Any]:
        """Fit a multivariate normal distribution to numeric data."""
        if self.df is None:
            raise ValueError("No data loaded.")
        cols = list(columns) if columns is not None else [c for c in self.df.select_dtypes(include=[np.number]).columns if c != "count"]
        X = self.df[cols].dropna().to_numpy(float)
        n, m = X.shape
        if n <= m:
            raise ValueError("n must be larger than number of features.")
        mu = X.mean(axis=0)
        S = np.cov(X, rowvar=False, ddof=1) + 1e-6 * np.eye(m)
        dist = multivariate_normal(mean=mu, cov=S, allow_singular=True)
        ll = float(np.sum(dist.logpdf(X)))
        k_params = m + (m * (m + 1)) // 2
        aic = float(2 * k_params - 2 * ll)
        print(f"Log-likelihood={ll:.4f}, AIC={aic:.4f}")
        return {"mean_vector": mu, "covariance_matrix": S, "log_likelihood": ll, "aic": aic,
                "distribution_object": dist, "features": cols}


    def instantiate_macro_clt_distribution(self, columns: Optional[Sequence[str]] = None) -> Dict[str, Any]:
        """Create CLT distribution for the sample mean vector: mean ~ N(mu_hat, S/n)."""
        if self.df is None:
            raise ValueError("No data loaded.")
        cols = list(columns) if columns is not None else [c for c in self.df.select_dtypes(include=[np.number]).columns if c != "count"]
        X = self.df[cols].dropna().to_numpy(float)
        n, m = X.shape
        if n <= m:
            raise ValueError("n must be larger than number of features.")
        mu = X.mean(axis=0)
        S = np.cov(X, rowvar=False, ddof=1)
        cov = S / n + 1e-10 * np.eye(m)
        dist = multivariate_normal(mean=mu, cov=cov, allow_singular=True)
        total_var = float(np.trace(cov))
        print(f"Total parameter variance Tr(S/n)={total_var:.8f}")
        return {"mean_vector": mu, "clt_covariance_matrix": cov, "total_parameter_variance": total_var,
                "distribution_object": dist, "features": cols}
    # ==========================================================================
    #  10. MODEL EVALUATION
    # ==========================================================================

    def evaluate_model(
        self,
        model=None,
        task: Optional[str] = None,
    ):
        """
        Evaluate a trained model on the held-out test set.
        Generates interactive Plotly plots appropriate to the task.

        task : 'classification' | 'regression' (auto-detected if None)
        """
        if self.X_test is None:
            print("❌ Run prepare_train_test() first.")
            return

        model = model or self._last_model
        if model is None:
            print("❌ Pass a model or run tune_model() first.")
            return

        task = task or self._task or "classification"
        model.fit(self.X_train, self.y_train)
        y_pred = model.predict(self.X_test)

        if task == "classification":
            self._eval_classification(model, y_pred)
        else:
            self._eval_regression(y_pred)

        self._last_model = model

    def _eval_classification(self, model, y_pred):
        acc = accuracy_score(self.y_test, y_pred)
        f1  = f1_score(self.y_test, y_pred, average="weighted")
        print(f"\n📊 Accuracy : {acc:.4f}  |  F1 (weighted): {f1:.4f}")
        print("\n" + classification_report(self.y_test, y_pred))

        # ── Confusion Matrix ──────────────────────────────────────────────────
        cm = confusion_matrix(self.y_test, y_pred)
        labels = np.unique(np.concatenate([self.y_test, y_pred]))
        fig_cm = px.imshow(
            cm, text_auto=True,
            x=[str(l) for l in labels], y=[str(l) for l in labels],
            color_continuous_scale="Blues",
            title="Confusion Matrix",
            labels={"x": "Predicted", "y": "Actual"},
        )
        fig_cm.show()

        # ── ROC curve (binary or multi-class) ────────────────────────────────
        try:
            classes = np.unique(self.y_test)
            if len(classes) == 2:
                if hasattr(model, "predict_proba"):
                    y_prob = model.predict_proba(self.X_test)[:, 1]
                elif hasattr(model, "decision_function"):
                    y_prob = model.decision_function(self.X_test)
                else:
                    return
                fpr, tpr, _ = roc_curve(self.y_test, y_prob)
                auc = roc_auc_score(self.y_test, y_prob)
                fig_roc = go.Figure()
                fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines",
                                             name=f"ROC (AUC={auc:.3f})",
                                             line=dict(color="royalblue", width=2)))
                fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                                             name="Random", line=dict(dash="dash", color="gray")))
                fig_roc.update_layout(title="ROC Curve", xaxis_title="FPR", yaxis_title="TPR")
                fig_roc.show()

                # Precision-Recall
                prec, rec, _ = precision_recall_curve(self.y_test, y_prob)
                fig_pr = go.Figure()
                fig_pr.add_trace(go.Scatter(x=rec, y=prec, mode="lines",
                                            line=dict(color="darkorange", width=2)))
                fig_pr.update_layout(title="Precision-Recall Curve",
                                     xaxis_title="Recall", yaxis_title="Precision")
                fig_pr.show()
        except Exception as e:
            print(f"⚠️  Could not plot ROC: {e}")

    def _eval_regression(self, y_pred):
        mae  = mean_absolute_error(self.y_test, y_pred)
        mse  = mean_squared_error(self.y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(self.y_test, y_pred)
        ev   = explained_variance_score(self.y_test, y_pred)
        print(f"\n📊 MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  ExplVar={ev:.4f}")

        # Actual vs Predicted
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=self.y_test, y=y_pred, mode="markers",
                                 marker=dict(opacity=0.5, color="steelblue"),
                                 name="Predictions"))
        mn = min(self.y_test.min(), y_pred.min())
        mx = max(self.y_test.max(), y_pred.max())
        fig.add_trace(go.Scatter(x=[mn, mx], y=[mn, mx], mode="lines",
                                 line=dict(color="red", dash="dash"), name="Perfect Fit"))
        fig.update_layout(title="Actual vs Predicted",
                          xaxis_title="Actual", yaxis_title="Predicted")
        fig.show()

        # Residuals
        residuals = self.y_test - y_pred
        fig2 = px.histogram(x=residuals, nbins=40, title="Residual Distribution",
                            labels={"x": "Residual"}, color_discrete_sequence=["indianred"])
        fig2.show()

    # ==========================================================================
    #  11. LEARNING CURVE
    # ==========================================================================

    def plot_learning_curve(
        self,
        model=None,
        cv: int = 5,
        scoring: str = "accuracy",
        train_sizes: Optional[np.ndarray] = None,
    ):
        """
        Interactive Plotly learning curve: training vs validation score vs dataset size.
        """
        if self.X_train is None:
            print("❌ Run prepare_train_test() first.")
            return
        model = model or self._last_model
        if model is None:
            print("❌ Pass a model or run tune_model() first.")
            return

        if train_sizes is None:
            train_sizes = np.linspace(0.1, 1.0, 8)

        sizes, train_scores, val_scores = learning_curve(
            model, self.X_train, self.y_train,
            cv=cv, scoring=scoring,
            train_sizes=train_sizes,
            n_jobs=-1,
        )

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=sizes,
            y=train_scores.mean(axis=1),
            error_y=dict(array=train_scores.std(axis=1)),
            mode="lines+markers", name="Train Score",
            line=dict(color="royalblue"),
        ))
        fig.add_trace(go.Scatter(
            x=sizes,
            y=val_scores.mean(axis=1),
            error_y=dict(array=val_scores.std(axis=1)),
            mode="lines+markers", name="Validation Score",
            line=dict(color="tomato"),
        ))
        fig.update_layout(
            title=f"Learning Curve ({scoring})",
            xaxis_title="Training Set Size",
            yaxis_title=scoring,
            template="plotly_white",
        )
        fig.show()

    # ==========================================================================
    #  12. CLUSTERING
    # ==========================================================================

    def cluster_data(
        self,
        columns: Optional[List[str]] = None,
        method: str = "kmeans",
        n_clusters: int = 3,
        eps: float = 0.5,
        min_samples: int = 5,
        show_elbow: bool = True,
    ) -> np.ndarray:
        """
        Cluster numeric columns.

        method     : 'kmeans' | 'dbscan' | 'agglomerative'
        n_clusters : used by kmeans / agglomerative
        eps        : DBSCAN neighbourhood radius
        show_elbow : plot elbow curve for KMeans (k=2..10)

        Returns array of cluster labels.
        """
        if self.df is None:
            return np.array([])
        targets = columns or [
            c for c in self.df.select_dtypes(include=[np.number]).columns
            if c != "count"
        ]
        X = self.df[targets].fillna(0).values
        X_scaled = StandardScaler().fit_transform(X)

        if method == "kmeans":
            if show_elbow:
                inertias = []
                k_range = range(2, min(11, len(X)))
                for k in k_range:
                    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
                    km.fit(X_scaled)
                    inertias.append(km.inertia_)
                fig = px.line(x=list(k_range), y=inertias, markers=True,
                              title="KMeans Elbow Curve",
                              labels={"x": "k (clusters)", "y": "Inertia"})
                fig.show()

            model = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
            labels = model.fit_predict(X_scaled)

        elif method == "dbscan":
            model = DBSCAN(eps=eps, min_samples=min_samples)
            labels = model.fit_predict(X_scaled)

        elif method == "agglomerative":
            model = AgglomerativeClustering(n_clusters=n_clusters)
            labels = model.fit_predict(X_scaled)
        else:
            raise ValueError(f"Unknown method '{method}'.")

        self.df["cluster"] = labels
        n_valid = len(set(labels) - {-1})

        if n_valid > 1:
            sil = silhouette_score(X_scaled[labels != -1], labels[labels != -1])
            dbi = davies_bouldin_score(X_scaled[labels != -1], labels[labels != -1])
            print(f"📐 Silhouette: {sil:.4f}  |  Davies-Bouldin: {dbi:.4f}")
        print(f"🔵 Clusters found: {n_valid}  (label -1 = noise in DBSCAN)")

        # 2-D PCA scatter of clusters
        if X_scaled.shape[1] >= 2:
            pca2 = PCA(n_components=2)
            Z2 = pca2.fit_transform(X_scaled)
            fig = px.scatter(
                x=Z2[:, 0], y=Z2[:, 1], color=labels.astype(str),
                title=f"Cluster Scatter ({method}) — PCA 2D projection",
                labels={"x": "PC1", "y": "PC2", "color": "Cluster"},
            )
            fig.show()

        return labels

    # ==========================================================================
    #  13. DIMENSIONALITY REDUCTION
    # ==========================================================================

    def reduce_dimensions(
        self,
        columns: Optional[List[str]] = None,
        method: str = "pca",
        n_components: int = 2,
        color_col: Optional[str] = None,
        perplexity: float = 30.0,
    ) -> np.ndarray:
        """
        Reduce dimensions and plot an interactive 2-D or 3-D scatter.

        method : 'pca' | 'tsne' | 'umap' (requires umap-learn)
        """
        if self.df is None:
            return np.array([])
        targets = columns or [
            c for c in self.df.select_dtypes(include=[np.number]).columns
            if c not in ("count", color_col or "")
        ]
        X = self.df[targets].fillna(0).values
        X_scaled = StandardScaler().fit_transform(X)

        if method == "pca":
            reducer = PCA(n_components=n_components)
            Z = reducer.fit_transform(X_scaled)
            ev = reducer.explained_variance_ratio_
            print(f"📊 PCA explained variance: {[f'{v:.3f}' for v in ev]}")

        elif method == "tsne":
            reducer = TSNE(n_components=n_components, perplexity=perplexity,
                           random_state=42, n_iter=1000)
            Z = reducer.fit_transform(X_scaled)

        elif method == "umap":
            if not _HAS_UMAP:
                print("⚠️  umap-learn not installed. Run: pip install umap-learn")
                return np.array([])
            reducer = umap.UMAP(n_components=n_components, random_state=42)
            Z = reducer.fit_transform(X_scaled)
        else:
            raise ValueError(f"Unknown method '{method}'.")

        color = self.df[color_col].astype(str) if color_col else None

        if n_components == 2:
            fig = px.scatter(
                x=Z[:, 0], y=Z[:, 1], color=color,
                title=f"{method.upper()} 2-D Projection",
                labels={"x": f"{method.upper()}-1", "y": f"{method.upper()}-2"},
                opacity=0.7,
            )
        else:
            fig = px.scatter_3d(
                x=Z[:, 0], y=Z[:, 1], z=Z[:, 2], color=color,
                title=f"{method.upper()} 3-D Projection",
                labels={"x": f"{method.upper()}-1", "y": f"{method.upper()}-2",
                        "z": f"{method.upper()}-3"},
                opacity=0.7,
            )
        fig.show()
        return Z

    # ==========================================================================
    #  14. SHAP EXPLANATIONS
    # ==========================================================================

    def explain_model_shap(self, model=None, max_display: int = 20):
        """
        Generate SHAP summary + waterfall plots (requires shap package).
        pip install shap
        """
        if not _HAS_SHAP:
            print("⚠️  shap not installed. Run: pip install shap")
            return
        if self.X_test is None:
            print("❌ Run prepare_train_test() first.")
            return

        model = model or self._last_model
        if model is None:
            print("❌ Pass a model or run tune_model() first.")
            return

        print("🔍 Computing SHAP values …")
        try:
            explainer = shap.TreeExplainer(model)
        except Exception:
            explainer = shap.KernelExplainer(model.predict, self.X_train[:100])

        shap_values = explainer.shap_values(self.X_test)
        shap.summary_plot(shap_values, self.X_test, max_display=max_display)

    # ==========================================================================
    #  15. VISUALIZATION HELPERS
    # ==========================================================================

    def plot_numerical(self, column_names: Union[str, List[str]]):
        """Violin + Scatter + Histogram for numeric columns."""
        if self.df is None:
            return
        if isinstance(column_names, str):
            column_names = [column_names]
        valid = [c for c in column_names
                 if c in self.df.columns and pd.api.types.is_numeric_dtype(self.df[c])]
        for col in valid:
            fig = make_subplots(rows=1, cols=3,
                                subplot_titles=(f"Violin: {col}", f"Scatter: {col}",
                                                f"Histogram: {col}"))
            fig.add_trace(go.Violin(x=self.df[col], box_visible=True,
                                    meanline_visible=True, name=col,
                                    orientation="h", line_color="lightseagreen"), row=1, col=1)
            fig.add_trace(go.Scatter(y=self.df[col], mode="markers",
                                     marker=dict(opacity=0.5, color="royalblue"), name=col),
                          row=1, col=2)
            fig.add_trace(go.Histogram(x=self.df[col], name=col,
                                        marker_color="indianred"), row=1, col=3)
            fig.update_layout(height=420, title_text=f"<b>{col}</b>",
                               showlegend=False, template="plotly_white")
            fig.show()

    def plot_categorical(self, column_names: Union[str, List[str]]):
        """Bar chart (count + %) for categorical columns."""
        if self.df is None:
            return
        if isinstance(column_names, str):
            column_names = [column_names]
        for col in column_names:
            counts = self.df[col].value_counts().reset_index()
            counts.columns = [col, "count"]
            counts["pct"] = (counts["count"] / counts["count"].sum() * 100).round(1).astype(str) + "%"
            fig = px.bar(counts, x=col, y="count", text="pct",
                         title=f"Frequency: {col}", color=col,
                         color_discrete_sequence=px.colors.qualitative.Pastel)
            fig.show()

    def plot_correlation_heatmap(self, method: str = "pearson"):
        """Interactive Pearson / Spearman / Kendall correlation heatmap."""
        if self.df is None:
            return
        num_df = self.df.select_dtypes(include=[np.number]).drop(columns=["count"], errors="ignore")
        corr = num_df.corr(method=method)
        fig = px.imshow(corr, text_auto=".2f", aspect="auto",
                        color_continuous_scale="RdBu_r",
                        title=f"{method.capitalize()} Correlation Heatmap")
        fig.show()

    def plot_pairplot(
        self,
        columns: Optional[List[str]] = None,
        color_col: Optional[str] = None,
    ):
        """Plotly scatter matrix (pair plot)."""
        if self.df is None:
            return
        num_cols = columns or [
            c for c in self.df.select_dtypes(include=[np.number]).columns
            if c != "count"
        ][:8]
        fig = px.scatter_matrix(
            self.df, dimensions=num_cols,
            color=color_col,
            title="Scatter Matrix (Pair Plot)",
            opacity=0.4,
        )
        fig.update_traces(diagonal_visible=False)
        fig.show()

    def plot_class_distribution(self, target: str):
        """Bar + Pie side-by-side showing class balance for a target column."""
        if self.df is None or target not in self.df.columns:
            return
        counts = self.df[target].value_counts().reset_index()
        counts.columns = [target, "count"]
        fig = make_subplots(rows=1, cols=2,
                            subplot_titles=("Class Counts", "Class Proportions"),
                            specs=[[{"type": "bar"}, {"type": "pie"}]])
        fig.add_trace(go.Bar(x=counts[target].astype(str), y=counts["count"],
                             marker_color=px.colors.qualitative.Plotly), row=1, col=1)
        fig.add_trace(go.Pie(labels=counts[target].astype(str),
                             values=counts["count"], hole=0.35), row=1, col=2)
        fig.update_layout(title=f"Class Distribution — '{target}'",
                          template="plotly_white", showlegend=False)
        fig.show()

    def plot_feature_importance(self, model, feature_names: List[str], top_n: int = 20):
        """Horizontal bar chart of a fitted model's feature importances."""
        if not hasattr(model, "feature_importances_"):
            print("⚠️  Model does not expose feature_importances_.")
            return
        imp = pd.Series(model.feature_importances_, index=feature_names).nlargest(top_n)
        fig = px.bar(imp.reset_index(), x=0, y="index", orientation="h",
                     title=f"Top-{top_n} Feature Importances",
                     labels={0: "Importance", "index": "Feature"},
                     color=0, color_continuous_scale="Viridis")
        fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=600)
        fig.show()

    # ==========================================================================
    #  16. EXPORT
    # ==========================================================================

    def export_model(self, model=None, filename: str = "model.pkl"):
        """Pickle a trained model and trigger a Colab download."""
        model = model or self._last_model
        if model is None:
            print("❌ No model to export.")
            return
        with open(filename, "wb") as f:
            pickle.dump(model, f)
        if _IN_COLAB:
            files.download(filename)
        print(f"💾 Model saved → '{filename}'")

    def export_cleaned_data(self, filename: str = "cleaned_data.csv"):
        """Download the current DataFrame as CSV."""
        if self.df is None:
            return
        self.df.to_csv(filename, index=False)
        if _IN_COLAB:
            files.download(filename)
        print(f"💾 Data saved → '{filename}'")

    def export_to_excel(self, filename: str = "data.xlsx"):
        """Download the current DataFrame as Excel."""
        if self.df is None:
            return
        self.df.to_excel(filename, index=False)
        if _IN_COLAB:
            files.download(filename)
        print(f"💾 Excel saved → '{filename}'")

    def export_to_parquet(self, filename: str = "data.parquet"):
        """Download the current DataFrame as Parquet."""
        if self.df is None:
            return
        self.df.to_parquet(filename, index=False)
        if _IN_COLAB:
            files.download(filename)
        print(f"💾 Parquet saved → '{filename}'")

    # ==========================================================================
    #  17. QUICK PIPELINE HELPER
    # ==========================================================================

    def quick_pipeline(
        self,
        target: str,
        task: str = "classification",
        scaler: str = "standard",
        model=None,
        cv: int = 5,
    ):
        """
        One-call convenience: encode → scale → split → benchmark → evaluate.

        Parameters
        ----------
        target : name of the target column
        task   : 'classification' | 'regression'
        scaler : 'standard' | 'minmax' | 'robust'
        model  : optional custom sklearn estimator to evaluate after benchmarking
        cv     : cross-validation folds
        """
        print("═" * 60)
        print("  🚀 Quick Pipeline Start")
        print("═" * 60)

        # Encode categoricals
        cat_cols = [c for c in self.df.select_dtypes(exclude=[np.number]).columns
                    if c not in ("count", target)]
        if cat_cols:
            self.encode_features(cat_cols, method="label")

        # Scale numerics (excluding target)
        num_cols = [c for c in self.df.select_dtypes(include=[np.number]).columns
                    if c not in ("count", target)]
        if num_cols:
            self.scale_features(num_cols, method=scaler)

        # Split
        self.prepare_train_test(target=target, task=task if task != "regression" else task)

        # Benchmark
        print(f"\n📊 Benchmarking {task} models …\n")
        if task == "classification":
            results = self.benchmark_classifiers(cv=cv)
        else:
            results = self.benchmark_regressors(cv=cv)

        # Evaluate best or provided model
        best_name = results.iloc[0]["Model"] if not results.empty else None
        if model is None and best_name:
            model_map_clf = {
                "Logistic Regression":  LogisticRegression(max_iter=1000),
                "Decision Tree":        DecisionTreeClassifier(),
                "Random Forest":        RandomForestClassifier(n_estimators=100),
                "Gradient Boosting":    GradientBoostingClassifier(),
                "AdaBoost":             AdaBoostClassifier(),
                "K-Nearest Neighbors":  KNeighborsClassifier(),
                "SVM (RBF)":            SVC(probability=True),
                "Naive Bayes":          GaussianNB(),
            }
            model_map_reg = {
                "Linear Regression":    LinearRegression(),
                "Ridge":                Ridge(),
                "Lasso":                Lasso(),
                "ElasticNet":           ElasticNet(),
                "Decision Tree":        DecisionTreeRegressor(),
                "Random Forest":        RandomForestRegressor(n_estimators=100),
                "Gradient Boosting":    GradientBoostingRegressor(),
                "AdaBoost":             AdaBoostRegressor(),
                "K-Nearest Neighbors":  KNeighborsRegressor(),
                "SVR":                  SVR(),
            }
            mm = model_map_clf if task == "classification" else model_map_reg
            model = mm.get(best_name)

        if model:
            print(f"\n🔍 Evaluating: {type(model).__name__} …")
            self.evaluate_model(model=model, task=task)
            self._last_model = model

        print("\n✅ Quick Pipeline Complete")


# ==============================================================================
#  USAGE EXAMPLES (run cell-by-cell in Colab)
# ==============================================================================

USAGE_GUIDE = """
╔══════════════════════════════════════════════════════════════╗
║              MLDataInspector — Quick-Start Guide             ║
╚══════════════════════════════════════════════════════════════╝

# ── 1. Install & import ──────────────────────────────────────
# (Upload this file to Colab, then:)
# from ml_data_inspector import MLDataInspector, install_optional

# ── 2. Create instance ───────────────────────────────────────
inspector = MLDataInspector()

# ── 3a. Load from Colab upload dialog (any supported format) ─
df = inspector.load_data()

# ── 3b. Load from a local / Drive path ──────────────────────
df = inspector.load_data('/content/drive/MyDrive/data.xlsx')
df = inspector.load_data('/content/data.json')
df = inspector.load_data('/content/data.parquet')

# ── 3c. Load from Google Sheets (must be publicly shared) ────
df = inspector.load_data(
    google_sheet_url='https://docs.google.com/spreadsheets/d/<ID>/edit#gid=0'
)

# ── 4. Explore ───────────────────────────────────────────────
inspector.get_summary()
inspector.column_details()
inspector.show_missing_data()

# ── 5. Clean ─────────────────────────────────────────────────
inspector.handle_missing_values(strategy='median')
inspector.remove_duplicates()
inspector.handle_outliers(method='iqr', action='remove')

# ── 6. Visualise ─────────────────────────────────────────────
inspector.plot_numerical(['age', 'income'])
inspector.plot_categorical(['gender', 'education'])
inspector.plot_correlation_heatmap()
inspector.plot_pairplot(color_col='target')
inspector.plot_class_distribution('target')

# ── 7. Feature engineering ───────────────────────────────────
inspector.encode_features(['gender', 'city'], method='onehot')
inspector.scale_features(method='standard')
inspector.add_log_features(['income', 'age'])

# ── 8. Feature selection ─────────────────────────────────────
top_feats = inspector.select_features('target', method='importance', k=10)

# ── 9. Prepare split ─────────────────────────────────────────
X_tr, X_te, y_tr, y_te = inspector.prepare_train_test('target')

# ── 10. Class balance (optional) ────────────────────────────
# install_optional('imbalanced-learn')
# X_tr, y_tr = inspector.balance_classes(X_tr, y_tr, method='smote')

# ── 11. Benchmark all models ────────────────────────────────
inspector.benchmark_classifiers(cv=5)

# ── 12. Tune best model ──────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
grid = inspector.tune_model(
    RandomForestClassifier(),
    {'n_estimators': [50, 100], 'max_depth': [None, 5, 10]},
)

# ── 13. Evaluate ────────────────────────────────────────────
inspector.evaluate_model(grid.best_estimator_, task='classification')
inspector.plot_learning_curve(grid.best_estimator_)

# ── 14. Cluster ──────────────────────────────────────────────
labels = inspector.cluster_data(method='kmeans', n_clusters=4)

# ── 15. Dimensionality reduction ────────────────────────────
Z = inspector.reduce_dimensions(method='tsne', n_components=2, color_col='target')

# ── 16. SHAP ────────────────────────────────────────────────
# install_optional('shap')
# inspector.explain_model_shap(grid.best_estimator_)

# ── 17. Export ──────────────────────────────────────────────
inspector.export_model(filename='best_model.pkl')
inspector.export_cleaned_data('cleaned.csv')
inspector.export_to_excel('cleaned.xlsx')

# ── 18. Or just run everything at once ──────────────────────
inspector.quick_pipeline(target='target', task='classification')
"""

if __name__ == "__main__":
    print(USAGE_GUIDE)